# CrashDiag hard-only GRPO on Kaggle

This independent notebook downloads a signed schema-v2 curriculum and its immutable SFT parent, calibrates sampling against the real Vultr sandbox, requires a nonzero-update smoke run, trains from the original SFT adapter, evaluates all 192 hard rows plus the original schema-v1 regression split, and only then promotes the run. No LLM grades success.

In [ ]:
from pathlib import Path
import gc
import json
import os
import re
import subprocess
import sys

REPO_URL = "https://github.com/Indium-AI-Labs/CrashDiag.git"
REPO_DIR = Path("/kaggle/working/CrashDiag")
BUCKET_ID = "devaanshpa/CrashDiag"
SANDBOX_URL = "https://sandbox.devaanshpathak.com"
HARD_RUN_ID = os.environ.get("CRASHDIAG_RUN_ID") or "PASTE_HARD_GRPO_RUN_ID_HERE"
SOURCE_COMMIT = os.environ.get("CRASHDIAG_SOURCE_COMMIT") or "PASTE_HARD_GRPO_SOURCE_COMMIT_HERE"
TRAINER_COMMIT = os.environ.get("CRASHDIAG_TRAINER_COMMIT") or "PASTE_TRAINER_COMMIT_HERE"
PRECISION = "auto"
SEED = 42
NUM_GENERATIONS = 8
SMOKE_STEPS = 36

if HARD_RUN_ID == "PASTE_HARD_GRPO_RUN_ID_HERE":
    raise ValueError("Set HARD_RUN_ID to GRPO_RUN_ID printed by training.generate_grpo_hard")
if re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT) is None:
    raise ValueError("Set SOURCE_COMMIT to the dataset SHA printed by training.generate_grpo_hard")
if re.fullmatch(r"[0-9a-f]{40}", TRAINER_COMMIT) is None:
    raise ValueError("Set TRAINER_COMMIT to the full hard-GRPO trainer SHA")
print(f"HARD_RUN_ID={HARD_RUN_ID}\nSOURCE_COMMIT={SOURCE_COMMIT}\nTRAINER_COMMIT={TRAINER_COMMIT}")

## Install the exact generator/trainer revision

In [ ]:
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"Refusing to overwrite {REPO_DIR}")
else:
    subprocess.run(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", TRAINER_COMMIT], check=True)
CURRENT_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if CURRENT_COMMIT != TRAINER_COMMIT:
    raise RuntimeError("trainer checkout mismatch")
# CrashDiag does not use it; Kaggle's old optional TorchAO can break current PEFT.
torchao_probe = subprocess.run([sys.executable, "-m", "pip", "show", "torchao"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
if torchao_probe.returncode == 0:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[train]"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"checked_out_trainer={CURRENT_COMMIT}")

## Load Kaggle Secrets and signed cross-run handoffs

In [ ]:
def required_secret(name: str) -> str:
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception as exc:
        raise RuntimeError(f"Attach Kaggle Secret {name!r}") from exc
    if not value:
        raise RuntimeError(f"Kaggle Secret {name!r} is empty")
    return value

os.environ["HF_TOKEN"] = required_secret("HF_TOKEN")
os.environ["CRASHDIAG_SANDBOX_TOKEN"] = required_secret("CRASHDIAG_SANDBOX_TOKEN")
os.environ["CRASHDIAG_SANDBOX_URL"] = SANDBOX_URL
os.environ["CRASHDIAG_HF_BUCKET_ID"] = BUCKET_ID
os.environ["CRASHDIAG_ARTIFACT_PREFIX"] = "runs"
os.environ["CRASHDIAG_ARTIFACT_LOCAL_ROOT"] = str(REPO_DIR / "artifacts")
os.environ["CRASHDIAG_ARTIFACT_UPLOAD_POLICY"] = "required"
os.environ["CRASHDIAG_RUN_ID"] = HARD_RUN_ID
print("Kaggle Secrets loaded; values are not printed.")

In [ ]:
from training.artifacts import ArtifactConfig, ArtifactUploader
from training.generate_grpo_hard import read_parent_reference

def make_uploader(run_id: str) -> ArtifactUploader:
    return ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=run_id, prefix="runs", policy="required", local_root=REPO_DIR / "artifacts", token=os.environ["HF_TOKEN"]))

def fetch_stage(client: ArtifactUploader, stage: str, target: Path, paths: list[str]) -> Path:
    if target.exists() and any(target.iterdir()):
        client.verify_local_stage(target, stage, include_paths=paths)
    else:
        client.download_stage(stage, target, include_paths=paths)
    return target

HANDOFF = Path("/kaggle/working/crashdiag-hard-handoff")
HARD_DATA = fetch_stage(make_uploader(HARD_RUN_ID), "datasets", REPO_DIR / "artifacts" / HARD_RUN_ID / "datasets", ["grpo_hard_train.jsonl", "grpo_hard_eval.jsonl", "grpo_hard_summary.json", "parent_sft.json"])
hard_manifest = json.loads((HARD_DATA / "manifest.json").read_text())
if hard_manifest.get("runtime", {}).get("git_commit") != SOURCE_COMMIT:
    raise RuntimeError("hard dataset/source commit mismatch")
hard_summary = json.loads((HARD_DATA / "grpo_hard_summary.json").read_text())
if hard_summary.get("curriculum_version") != 2 or hard_summary.get("action_contract") != "parameter_free_repairs":
    raise RuntimeError("hard dataset is not the corrected curriculum-v2 action contract")
parent_pointer = json.loads((HARD_DATA / "parent_sft.json").read_text())
PARENT_SFT_RUN_ID = parent_pointer["run_id"]
parent_client = make_uploader(PARENT_SFT_RUN_ID)
PARENT_SFT_DIR = fetch_stage(parent_client, "sft", HANDOFF / PARENT_SFT_RUN_ID / "sft", ["adapter_config.json", "adapter_model.safetensors", "chat_template.jinja", "tokenizer.json", "tokenizer_config.json"])
PARENT_DATA = fetch_stage(parent_client, "datasets", HANDOFF / PARENT_SFT_RUN_ID / "datasets", ["grpo_eval.jsonl"])
verified_parent = read_parent_reference(PARENT_SFT_DIR, PARENT_SFT_RUN_ID)
for key in ("manifest_sha256", "adapter_sha256", "base_model"):
    if verified_parent[key] != parent_pointer[key]:
        raise RuntimeError(f"parent SFT handoff mismatch: {key}")
uploader = make_uploader(HARD_RUN_ID)
print(f"verified curriculum v2 hard run and parent SFT run {PARENT_SFT_RUN_ID}")

## Probe schema-v2 Vultr service and Kaggle GPU

In [ ]:
from crashdiag.sandbox_apps.http import HttpSandbox
import torch

with HttpSandbox(SANDBOX_URL, api_token=os.environ["CRASHDIAG_SANDBOX_TOKEN"], timeout=15.0) as sandbox:
    service = sandbox.service_health()
    application = sandbox.health_check()
if 2 not in service.get("scenario_schema_versions", []):
    raise RuntimeError(f"Vultr service lacks schema v2: {service}")
if service.get("hard_scenario_batch") is not True:
    raise RuntimeError("Redeploy Vultr: atomic hard-scenario setup is required for calibration")
if application.get("healthy") is not True:
    raise RuntimeError(f"sandbox preflight failed: {application}")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU before continuing")
print(f"gpu={torch.cuda.get_device_name(0)}, bf16={torch.cuda.is_bf16_supported()}")

## Calibrate broad 8-way sampling; reuse its immutable signed stage on rerun

In [ ]:
from training.calibrate_grpo import main as calibrate_main

SOURCE_ROLLOUT_STAGE = "calibration-contract-v2"
SOURCE_ROLLOUT_CACHE = fetch_stage(uploader, SOURCE_ROLLOUT_STAGE, HANDOFF / HARD_RUN_ID / SOURCE_ROLLOUT_STAGE, ["calibration.json", "rollouts.jsonl"])
CALIBRATION_STAGE = "calibration-declarative-v3-live"
CALIBRATION_OUTPUT = REPO_DIR / "outputs/grpo-hard-calibration-declarative-v3-live"
CALIBRATION_CACHE = HANDOFF / HARD_RUN_ID / CALIBRATION_STAGE
if uploader.stage_is_complete(CALIBRATION_STAGE):
    CALIBRATION_DIR = fetch_stage(uploader, CALIBRATION_STAGE, CALIBRATION_CACHE, ["calibration.json", "calibration.svg"])
    print(f"Reusing completed immutable stage: {CALIBRATION_STAGE}")
else:
    CALIBRATION_DIR = CALIBRATION_OUTPUT
    calibration_exit = calibrate_main([
        "--model", str(PARENT_SFT_DIR),
        "--train-file", str(HARD_DATA / "grpo_hard_train.jsonl"),
        "--output-dir", str(CALIBRATION_DIR),
        "--temperatures", "1.5", "1.6", "1.7",
        "--top-p", "0.9",
        "--top-k", "50",
        "--num-generations", str(NUM_GENERATIONS),
        "--prompts-per-fault-profile", "2",
        "--reward-workers", "8",
        "--max-new-tokens", "48",
        "--reuse-rollouts", str(SOURCE_ROLLOUT_CACHE / "rollouts.jsonl"),
        "--reuse-report", str(SOURCE_ROLLOUT_CACHE / "calibration.json"),
        "--source-rollout-stage", SOURCE_ROLLOUT_STAGE,
        "--precision", PRECISION,
        "--artifact-stage", CALIBRATION_STAGE,
    ])
    if calibration_exit not in (0, 3):
        raise RuntimeError(f"calibration process failed with status {calibration_exit}")
calibration = json.loads((CALIBRATION_DIR / "calibration.json").read_text())
if calibration.get("passed") is not True:
    raise RuntimeError(f"{CALIBRATION_STAGE} did not find usable mechanical reward variance; full GRPO remains blocked")
SELECTED_TEMPERATURE = float(calibration["selected_temperature"])
SELECTED_TOP_P = float(calibration["sampling"]["top_p"])
SELECTED_TOP_K = int(calibration["sampling"]["top_k"])
from IPython.display import SVG, display
print(json.dumps(calibration, indent=2, sort_keys=True))
display(SVG(filename=str(CALIBRATION_DIR / "calibration.svg")))

## Run the mandatory 36-step nonzero-update smoke job

In [ ]:
from training.grpo import main as grpo_main

SMOKE_DIR = REPO_DIR / "outputs/grpo-hard-smoke"
smoke_args = [
    "--model", str(PARENT_SFT_DIR),
    "--train-file", str(HARD_DATA / "grpo_hard_train.jsonl"),
    "--eval-file", "",
    "--output-dir", str(SMOKE_DIR),
    "--artifact-stage", "grpo-hard-smoke",
    "--precision", PRECISION,
    "--batch-size", "1",
    "--gradient-accumulation-steps", "8",
    "--num-generations", str(NUM_GENERATIONS),
    "--max-steps", str(SMOKE_STEPS),
    "--epochs", "1",
    "--learning-rate", "5e-6",
    "--temperature", str(SELECTED_TEMPERATURE),
    "--top-p", str(SELECTED_TOP_P),
    "--top-k", str(SELECTED_TOP_K),
    "--beta", "0.02",
    "--save-steps", "18",
    "--seed", str(SEED),
    "--parent-reference", str(HARD_DATA / "parent_sft.json"),
    "--require-nonzero-update",
    "--minimum-gate-steps", str(SMOKE_STEPS),
    "--report-to", "none",
]
grpo_main(smoke_args)
smoke_gate = json.loads((SMOKE_DIR / "reports/smoke_gate.json").read_text())
if smoke_gate.get("passed") is not True:
    raise RuntimeError("smoke gate failed; full GRPO must not start")
print(json.dumps(smoke_gate, indent=2, sort_keys=True))
for chart in sorted((SMOKE_DIR / "reports").glob("*.svg")):
    display(SVG(filename=str(chart)))

## Train hard GRPO from the original SFT adapter, not from smoke weights

In [ ]:
gc.collect()
torch.cuda.empty_cache()
FULL_DIR = REPO_DIR / "outputs/grpo-hard"
full_args = [
    "--model", str(PARENT_SFT_DIR),
    "--train-file", str(HARD_DATA / "grpo_hard_train.jsonl"),
    "--eval-file", "",
    "--output-dir", str(FULL_DIR),
    "--artifact-stage", "grpo-hard",
    "--precision", PRECISION,
    "--batch-size", "1",
    "--gradient-accumulation-steps", "8",
    "--num-generations", str(NUM_GENERATIONS),
    "--max-steps", "-1",
    "--epochs", "1",
    "--learning-rate", "5e-6",
    "--temperature", str(SELECTED_TEMPERATURE),
    "--top-p", str(SELECTED_TOP_P),
    "--top-k", str(SELECTED_TOP_K),
    "--beta", "0.02",
    "--save-steps", "24",
    "--seed", str(SEED),
    "--report-to", "none",
]
grpo_main(full_args)
for chart in sorted((FULL_DIR / "reports").glob("*.svg")):
    display(SVG(filename=str(chart)))

## Evaluate all hard rows and the original schema-v1 regression split

In [ ]:
gc.collect()
torch.cuda.empty_cache()
from training.evaluate_jsonl import main as evaluate_jsonl_main

HARD_EVAL_DIR = REPO_DIR / "outputs/hard-evaluation"
REGRESSION_DIR = REPO_DIR / "outputs/regression-evaluation"
evaluate_jsonl_main([
    "--model", str(FULL_DIR),
    "--dataset", str(HARD_DATA / "grpo_hard_eval.jsonl"),
    "--output-dir", str(HARD_EVAL_DIR),
    "--precision", PRECISION,
    "--artifact-stage", "hard-evaluation",
])
gc.collect()
torch.cuda.empty_cache()
evaluate_jsonl_main([
    "--model", str(FULL_DIR),
    "--dataset", str(PARENT_DATA / "grpo_eval.jsonl"),
    "--output-dir", str(REGRESSION_DIR),
    "--precision", PRECISION,
    "--artifact-stage", "regression-evaluation",
])
for report_dir in (HARD_EVAL_DIR / "reports", REGRESSION_DIR / "reports"):
    print(json.dumps(json.loads((report_dir / "mechanical_evaluation_summary.json").read_text()), indent=2, sort_keys=True))
    for chart in sorted(report_dir.glob("*.svg")):
        display(SVG(filename=str(chart)))

## Promote only after hard and regression gates pass

In [ ]:
from training.grpo_gates import promotion_gate, require_passed, write_gate

promotion = promotion_gate(HARD_EVAL_DIR / "mechanical_evaluation.json", REGRESSION_DIR / "mechanical_evaluation.json")
PROMOTION_DIR = REPO_DIR / "outputs/promotion"
promotion_path = write_gate(PROMOTION_DIR / "promotion_gate.json", promotion)
uploader.start_stage("promotion", {"scoring": "mechanical_fault_resolution"})
uploader.upload_files([promotion_path], "promotion", metadata=promotion)
print(json.dumps(promotion, indent=2, sort_keys=True))
require_passed(promotion, "hard GRPO promotion")
uploader.complete_run({"stages": ["datasets", CALIBRATION_STAGE, "grpo-hard-smoke", "grpo-hard", "hard-evaluation", "regression-evaluation", "promotion"], "parent_sft_run_id": PARENT_SFT_RUN_ID})
print(f"PROMOTED={uploader.remote_uri()}")